# Lab 1: The Apollo Missions

## Introduction
------

Imagine you are an engineer working on the very early days of the Apollo program. In order to
send people to the moon, you need a basic understanding of the gravitational potential and forces
the mission will experience. You also need to start quantifying the performance of the new rocket
that will carry the capsule: the Saturn V.

Your team at NASA has been tasked with conducting a series of investigations, which you are guided through below.
You will make some plots, and also write some short paragraphs with your scientific conclusions.
The NASA Director Gene Kranz will use your notebook to defend NASA funding to congress.
So your answers should be short and to the point, quantitative in nature, 
and contain highly polished plots, with appropriate titles, units, etc.

**Other important requirements:**
- Your final notebook---BOTH the .ipynb file AND the html export---should be COMMITTED and PUSHED to your fork of the PHYS265-spring26 GitHub repository. Instructions for doing this are available [online]( https://github.com/astroumd/PHYS265-spring26/blob/main/github.pdf).
- All plotting should be done in object oriented fashion, as discussed in class. 
Any use of “`plt.plot()`” will result in zero points being awarded for that question.
- Before submitting your work, reset JLD and then re-run your final code (Kernel → Restart Kernel and Run All Cells…) Make sure your code executes as intended and without error. The first cell should be "1", and the cells should execute in order.

A table of relevant physical constants is below. Good luck!

| Quantity | Value |
|----------|-------|
| Gravitational Constant ($G$) | $6.67 \times 10^{-11}$ m³/kg/s² |
| Gravitational Acceleration ($g$) | 9.81 m/s² |
| Mass of the Earth ($M_E$) | $5.9 \times 10^{24}$ kg |
| Mass of the Moon ($M_M$) | $7.3 \times 10^{22}$ kg |
| Mass of Apollo Command Module | 5500 kg |
| Radius of the Earth ($R_E$) | 6378 km |
| Radius of the Moon ($R_M$) | 1737 km |
| Distance from Earth to Moon ($d_{EM}$) | $3.8 \times 10^{8}$ m |
| Exhaust Velocity of Saturn V Stage 1 (S-1C) ($v_e$) | $2.4 \times 10^{3}$ m/s |
| Burn Rate of S-1C ($\dot{m}$) | $1.3 \times 10^{4}$ kg/s |
| Wet Mass of S-1C ($m_0$) | $2.8 \times 10^{6}$ kg |
| Dry Mass of S-1C ($m_f$) | $7.5 \times 10^5$ kg |

In [ ]:
#show
# In the code cell beneath this one you can put your imports,
# and define any variables you might need (e.g. the mass of the earth)

In [ ]:
import numpy as np
import scipy.constants as sc #just importing scipy constants in order to be able to get permittivity of free space
import matplotlib.pyplot as plt
#makes plots interactive! 
%matplotlib ipympl 
import matplotlib.colors as colors
import math

import matplotlib.ticker as ticker
#cbar log ticks

from scipy.integrate import quad
from scipy.integrate import dblquad
from scipy.integrate import solve_ivp
#these were imported as I tried using the integral definition to solve for the potential but I didnt like it so this is all that remains

from matplotlib.colors import LogNorm
#pulling these in to make log plots and adding circles for the earths surface in easier
from matplotlib.patches import Circle

from matplotlib.lines import Line2D
#for legends

#trying to get around using isInstance a million times I found Python has a typing library
from typing import Union, Tuple

Number = Union[int, float]


G = 6.67e-11           #m^3/kg/s^2
g = 9.81               #earths gravity constant m/s^2
massEarth = 5.9e24     #kg
massMoon = 7.3e22      #kg
massApolloCM = 5500    #kg
radiusEarth = 6378000  #m
radiusMoon = 1737000   #m
avgEtoM = 3.8e8        #m
exVelSatV = 2.4e3      #m
burnRateSV = 1.3e4     #kg/s
wetMSV = 2.8e6         #kg
dryMSV = 7.5e5         #kg

ArrayLike = Union[float, int, Tuple, np.ndarray]

## Part 1: The Gravitational Potential of the Earth
----

The gravitational potential at a distance $r$ from a mass $M$ is given by:

$$\Phi(r) = -\frac{GM}{r}$$

where G is the gravitational constant.

1. Code up a function that takes as an input the mass of a body $M$ , the cartesian location of the
mass $(x_M , y_M)$, and the location of some distant evaluation point $(x, y)$. Have the function
return the potential at the evaluation point. You may consider approximating the Earth as
a point particle, but in which case, be careful! For in that approximation, the the potential
contains a singularity. In our function, use all modern best practices for coding.
E.g. type hints, type checks, pre and post conditions, doc strings, etc.

In [ ]:
# James Watt 2026 UMD PHYS 265 LAB 1

def gravPotential2(M: float, xm: float, ym: float, x: ArrayLike, y: ArrayLike, R: float) -> np.ndarray:
    """
    Compute the gravitational potential at Cartesian locations (x, y)
    due to a spherically symmetric body of total mass M and radius R
    centered at (xm, ym).

    The body is assumed to have uniform density.

    Support NumPy arrays and float inputs

    Parameters
    ----------
    M : float
        Total mass of the body (kg). Must be positive.
    xm : float
        x-coordinate of the body's center (m).
    ym : float
        y-coordinate of the body's center (m).
    x : float or np.ndarray
        x-coordinate(s) of evaluation point(s) (m).
    y : float or np.ndarray
        y-coordinate(s) of evaluation point(s) (m).
    R : float
        Radius of the body (m). Must be positive.

    Outputs
    -------
    np.ndarray
        Gravitational potential at (x, y) in J/kg.
        Shape follows NumPy broadcasting rules.

    Raises
    ------
    TypeError
        If scalar parameters are not numeric.
    ValueError
        If M <= 0 or R <= 0.
    """

    # -------- Scalar type checks --------
    scalar_inputs = [M, xm, ym, R]
    if not all(isinstance(val, (int, float, np.floating)) for val in scalar_inputs):
        raise TypeError("M, xm, ym, and R must be numeric scalars.")

    if M <= 0:
        raise ValueError("Mass M must be positive.")
    if R <= 0:
        raise ValueError("Radius R must be positive.")

    # Convert evaluation points to arrays (supports scalar or array input)
    x_arr = np.asarray(x, dtype=float)
    y_arr = np.asarray(y, dtype=float)

    # -------- Distance from center --------
    dx = x_arr - xm
    dy = y_arr - ym
    r = np.hypot(dx, dy)

    # ----- Allocate output array -----
    V = np.empty_like(r, dtype=float)

    # Creates an array of the same shape as r, where each element is: True if that point is outside the sphere OR False if that point is inside the sphere
    outside = r >= R
    inside = ~outside
    #Allows us to use both formulas correctly
    
    # ----- Outside sphere -----
    V[outside] = -G * M / r[outside]

    # ----- Inside uniform sphere -----
    V[inside] = -G * M * (3 * R**2 - r[inside]**2) / (2 * R**3)

    # ----- Post-condition: ensure finite -----
    if not np.all(np.isfinite(V)):
        raise ArithmeticError("Computed potential contains non-finite values.")

    return V

2. Make a 1D plot of the absolute value of the gravitational potential $|\Phi|$ as a function of distance $x$ from the surface of the Earth, assuming $y = 0$.
Plot out to $1.5\times$ the distance from the Earth to the Moon. Because the potential falls quickly with r, make the y-axis logarithmic to visualize it better.

In [ ]:
#______Distance range______
x_max = 1.5 * avgEtoM

# ----- Distance from surface -----
x_surface = np.linspace(0, x_max, 2000)

# ----- Evaluation coordinate arrays(Earth's Surface at origin, y = 0) -----
x_eval = radiusEarth + x_surface
y_eval = np.zeros_like(x_eval)
#since I wrote my ggravPotential2 to consider earth as a sphere, not point mass, I have to set x=surface of the earth. At 0 my function will use a diferent forumula to solve for the graviational potential

# ----- Compute potential -----
V = gravPotential2(M=massEarth,xm=0.0,ym=0.0,x=x_eval,y=y_eval,R=radiusEarth)
#V = gravPotential2(M=massEarth,xm=0.0,ym=0.0,x=x_surface,y=0.0,R=radiusEarth)

V_abs = np.abs(V)

#______Plot______
fig12, ax12 = plt.subplots()
ax12.plot(x_surface, V_abs)
ax12.set_yscale("log")
ax12.set_xlabel("Distance from Earth's surface (m)")
ax12.set_ylabel("|Gravitational Potential| (J/kg)")
ax12.set_title("Gravitational Potential vs Distance from Earth's Surface")
ax12.grid()

3. Then, make a 2D color-mesh plot of the potential $\Phi$ with the Earth at the origin. Plot $|\Phi|$
over a large range of $x$ and $y$, for example, from $-1.5d_{EM}$ to $+1.5d_{EM}$ . Force the plot to have
a square aspect ratio. Make sure to add a nice colorbar, and make the colorbar logarithmic.

In [ ]:
#----- Creating Data Domain and Range & Loop for Potential Calculation-----
extent = 1.5 * avgEtoM
N = 1000 #grid resolution


# ----- Create 2D grid -----
x = np.linspace(-extent, extent, N)
y = np.linspace(-extent, extent, N)
X, Y = np.meshgrid(x, y)

# ----- Compute gravitational potential -----
V = gravPotential2(M=massEarth,xm=0.0,ym=0.0,x=X,y=Y,R=radiusEarth)

V_abs = np.abs(V)
#V_abs = V

# _____Plottin_____
fig13, ax13 = plt.subplots()

# ----- Colormesh with log norm -----
mesh = ax13.pcolormesh(X, Y, V_abs, norm=LogNorm(vmin=V_abs.min(), vmax=V_abs.max()),cmap="gist_earth", shading="auto")

# ----- Overlay Earth's boundary -----
earth_circle = Circle((0.0, 0.0),radiusEarth,fill=False,linewidth=1)

ax13.add_patch(earth_circle)

# ----- Force square aspect ratio & set bounds -----
ax13.set_aspect('equal', adjustable='box')
ax13.set_xlim(-extent,extent)
ax13.set_ylim(-extent,extent)

# ----- Labels and title ------
ax13.set_xlabel("x (m)")
ax13.set_ylabel("y (m)")
ax13.set_title("Gravitational Potential Magnitude Around Earth")

# ----- Add logarithmic colorbar ------
cbar = fig13.colorbar(mesh, ax=ax13)
cbar.set_label("|Gravitational Potential| (J/kg)")

cbar.locator = ticker.LogLocator()
cbar.formatter = ticker.LogFormatterMathtext()
cbar.update_ticks()

## Part 2: The Gravitational Potential of the Earth-Moon System
-----

Now, imagine the Earth is at the origin $(x_{E},y_{E}) = (0,0)$, 
and the moon is at a location $(x_{M},y_{M}) = \big( \frac{d_{EM}}{\sqrt{2}}, \frac{d_{EM}}{\sqrt{2}} \big)$.

1. Make an updated 2D color-mesh plot, from $-1.5d_{EM}$ to $+1.5d_{EM}$,
of the gravitational potential of the combined Earth-Moon system.

In [ ]:
moonX = avgEtoM/np.sqrt(2)
moonY = avgEtoM/np.sqrt(2)

# --- Compute potentials ---
V_earth = gravPotential2(massEarth, 0.0, 0.0, X, Y, radiusEarth)

V_moon = gravPotential2(massMoon, moonX, moonY, X, Y, radiusMoon)

V_total = V_earth + V_moon
V_abs = np.abs(V_total)

#______2D LOG COLOR-MESH PLOT______

fig1, ax1 = plt.subplots()

norm = colors.LogNorm(vmin=V_abs.min(), vmax=V_abs.max())
#should have done this earlier but was too lazy

mesh = ax1.pcolormesh(X, Y, V_abs,shading="auto",cmap="gist_earth",norm=norm)

# ----- Add Earth and Moon boundaries -----
earth_circle = Circle((0, 0), radiusEarth, fill=False, linewidth=1, color="blue")
moon_circle = Circle((moonX, moonY), radiusMoon, fill=False, linewidth=1, color="white")

ax1.add_patch(earth_circle)
ax1.add_patch(moon_circle)

ax1.set_aspect("equal")
ax1.set_xlim(-extent, extent)
ax1.set_ylim(-extent, extent)

ax1.set_title("Log-Scale Gravitational Potential Magnitude\nEarth-Moon System")
ax1.set_xlabel("x (m)")
ax1.set_ylabel("y (m)")

cbar1 = fig1.colorbar(mesh, ax=ax1)
cbar1.set_label("|Gravitational Potential| (J/kg)")
cbar1.locator = ticker.LogLocator()
cbar1.formatter = ticker.LogFormatterMathtext()
cbar1.update_ticks()

2. Additionally, make a 2D contour plot over the same $x$ and $y$ range.
Do not let all the contours bunch up around the Earth and Moon.
You will probably again need to tell matplotlib to use a logarithmic color scale.

In [ ]:
#______LOG-CONTOUR PLOT (well spaced)______

fig2, ax2 = plt.subplots()

# ----- Log spaced contour levels -----
levels = np.logspace(np.log10(V_abs.min()), np.log10(V_abs.max()), 16)

contours = ax2.contour(X, Y, V_abs,levels=levels,norm=norm,cmap="BrBG")

# ----- Add bodies -----
ax2.add_patch(Circle((0, 0), radiusEarth, fill=False, linewidth=10, color="blue"))
ax2.add_patch(Circle((moonX, moonY), radiusMoon, fill=False, linewidth=10, color="grey"))

ax2.set_aspect("equal")
ax2.set_xlim(-extent, extent)
ax2.set_ylim(-extent, extent)

ax2.set_title("Log-Spaced Gravitational Potential Contours\nEarth-Moon System")
ax2.set_xlabel("x (m)")
ax2.set_ylabel("y (m)")

ax2.clabel(contours, inline=True, fontsize=4)

## Part 3: The Gravitational Force Field of the Earth-Moon System
----
The gravitational force $\vec{F}$ 
that a mass $M_1$ exerts on a mass $m_2$ is:

$$\vec{F}_{21} = -G \frac{M_1 m_2}{|\vec{r}_{21}|^2} \hat{r}_{21}$$

where $\vec{r}_{21}$ is the displacement vector from $M_1$ to $m_2$.

1. Code up a function that takes as an input the mass of a body $M_1$, 
the mass of a secondary body $m_2$,
the cartesian location of the first mass $(x_1, y_1)$, 
and the cartesian location of the second body $(x_2, y_2)$. 
Have the function return the two gravitational force components, 
$F_x$ and $F_y$, experienced by the mass $m_2$ 
from the force exerted by $M_1$.
Note that the gravitational force contains a singularity, 
and so if you made the approximation that the Earth is a finite point,
you will need to avoid it.
Like for the 1D potential, make sure it follows modern best-coding practices.

In [ ]:
 def gravForce(M1: float, m2: float, x1: float, y1: float, x2: ArrayLike, y2: ArrayLike, epsilon: float = 1e3) -> Tuple[np.ndarray, np.ndarray]:
    """
    Compute the gravitational force components exerted by mass M1
    located at (x1, y1) on mass m2 located at (x2, y2).

    Supports scalar or NumPy array inputs for x2 and y2.

    Parameters
    ----------
    M1 : float
        Mass of the primary body (kg)
    m2 : float
        Mass of the secondary body experiencing the force (kg)
    x1, y1 : float
        Cartesian position of M1 (meters)
    x2, y2 : float or np.ndarray
        Position(s) of m2 (meters)


    Outputs
    -------
    Fx, Fy : np.ndarray
        Force components (Newtons) exerted on m2
    """

    # ----- Type checks -----
    if not all(isinstance(v, (int, float, np.floating)) for v in [M1, m2, x1, y1]):
        raise TypeError("M1, m2, x1, and y1 must be numeric scalars")

    if M1 <= 0 or m2 <= 0:
        raise ValueError("Masses must be positive!")

    # ----- Convert to arrays -----
    x2 = np.asarray(x2, dtype=float)
    y2 = np.asarray(y2, dtype=float)

    # ----- Displacement vector ------
    dx = x2 - x1
    dy = y2 - y1

    # ----- Distance -----
    r2 = dx**2 + dy**2
    r = np.sqrt(r2)

    # ----- Force magnitude factor ------
    factor = -G * M1 * m2 / r**3

    Fx = factor * dx
    Fy = factor * dy

    # ----- Post-conditions------
    if not (np.all(np.isfinite(Fx)) and np.all(np.isfinite(Fy))):
        raise ArithmeticError("Non-finite force encountered")

    return Fx, Fy

2. Assuming the Earth is still at the origin and the Moon is still at 
$(x_M, y_M) = \big( \frac{d_{EM}}{\sqrt{2}}, \frac{d_{EM}}{\sqrt{2}} \big)$, 
make a 2D streamplot of the gravitational force the Earth-Moon system 
will exert on the Apollo 11 command module.
Plot the force over the same x-y range of $-1.5 d_{EM}$ to $+1.5 d_{EM}$.
Add a nice color bar, which will again probably need to be logarithmic.
Mark the location of the earth with a black circle, and the moon with a black square.
Add a legend.

In [ ]:
# ----- Grid -----
N = 2500

x = np.linspace(-extent, extent, N)
y = np.linspace(-extent, extent, N)

X, Y = np.meshgrid(x, y)

#______FORCES_______

# ----- Force from Earth -----
Fx_E, Fy_E = gravForce(massEarth, massApolloCM, 0.0, 0.0, X, Y)

# ----- Force from Moon -----
Fx_M, Fy_M = gravForce(massMoon, massApolloCM, moonX, moonY, X, Y)

# ----- Total force -----
Fx = Fx_E + Fx_M
Fy = Fy_E + Fy_M

# ----- Force magnitude -----
Fmag = np.sqrt(Fx**2 + Fy**2)

#______Plot______
fig32, ax32 = plt.subplots()
norm = colors.LogNorm(vmin=Fmag.min(), vmax=Fmag.max())
stream = ax32.streamplot(X, Y, Fx, Fy, color=Fmag, cmap="terrain", norm=norm, density=2)

# ----- Earth marker -----
earth = Circle((0,0), radiusEarth, color="black", fill=False, linewidth=4)
ax32.add_patch(earth)

# ----- Moon marker -----
ax32.scatter(moonX, moonY, color="black", marker="s", s=10)

# ----- Formatting -----
ax32.set_aspect("equal")
ax32.set_xlim(-extent, extent)
ax32.set_ylim(-extent, extent)

ax32.set_xlabel("x (m)")
ax32.set_ylabel("y (m)")
ax32.set_title("Gravitational Force Field of the Earth–Moon System")

# ----- Colorbar -----
cbar = fig32.colorbar(stream.lines, ax=ax32)
cbar.set_label("Force Magnitude (N)")

# ----- Legend -----
legend_elements = [
    Line2D([0],[0],marker='o',color='black',label='Earth',markersize=10,linestyle='None'), #earth
    Line2D([0],[0],marker='s',color='black',label='Moon',markersize=10,linestyle='None')   #moon
]

ax32.legend(handles=legend_elements)

3. You will note there is a point between the Earth and the Moon where the field lines seem to vanish. This is the point where the Earth's and Moon's gravitational forces cancel eachother. By setting $|\vec{F}_{moon}| = |\vec{F}_{earth}|$, calculate the radial distance of the "balance point." From there, identify the $x$ and $y$ coordinates of the balance point. Typeset your derivation (show multiple steps!) in latex in a markdown cell below. Then, calculate it numerically, and draw the answer onto a new version of your force field plot. Again mark the location of the earth with a black circle, and the moon with a black square; mark the location of the balance point with a red x. Add a legend.

**Your Derivation Goes in the markdown cell below:**

And your updated plot, including the balance point, goes in the cell below.

In [ ]:
# balance point distance from Earth
r_balance = avgEtoM * np.sqrt(massEarth) / (np.sqrt(massEarth) + np.sqrt(massMoon))

x_balance = r_balance / np.sqrt(2)
y_balance = r_balance / np.sqrt(2)

ax32.scatter(x_balance, y_balance, color="red", s=120, label="Balance Point")
fig32.show()

4. Quantitatively discuss your conclusions. Is the balance point close to the Moon or Earth? How close? Does this make sense?

**Your Answer Goes in the markdown cell below:**

Starting with Newton's gravitational force 

$$ F = G \left(\frac{Mm}{r^2}\right) $$

At the Balance point:

$$ |F_{moon} | = |F_{earth} | $$

$$ G \left(\frac{M_{earth}m}{r^2}\right) = G \left(\frac{M_{moon}m}{(d-r)^2}\right) $$

G and m are shared constants so they all cancel and take square root of both sides to simplfy:

$$ \left(\frac{\sqrt{M_{earth}}}{r}\right) = \left(\frac{\sqrt{M_{moon}}}{d-r}\right) $$ 

Rearrange:

$$ r\sqrt{M_{moon}} = (d-r)\sqrt{M_{earth}} $$

Now solve for r to get the distance from Earth to the balance point:

$$ r = \left(\frac{d\sqrt{M_{earth}}}{\sqrt{M_{earth}}+\sqrt{M_{moon}}}\right) $$

## Part 4: Altitude of the Saturn V Rocket
----
Rockets function by conservation of momentum.
The ejection of fuel backwards propels the rocket forwards.
The change in a rocket's velocity $\Delta v$ as a 
function of time $t$ (under a set of simplifying assumptions) 
is given by the Tsiolkovsky rocket equation:

$$\Delta v(t) = v_e \ln \left(\frac{m_0}{m(t)}\right) - gt$$

where $m_0$ is the initial "wet" mass (fuel + rocket parts + payload), 
$m(t) = m_0 - \dot{m}t$ is the mass at time $t$, 
$\dot{m}$ is the fuel burn rate (which we will assume to be constant), 
$v_e$ is the fuel exhaust velocity, and $g$ is the gravitational acceleration.

You can find the altitude $h$ of the rocket at "burnout" 
(when all the fuel is used up) by integrating the velocity over time:

$$h = \int_{0}^{T} \Delta v(t) \, dt$$

where $T$ is the total burn time of the rocket, 
which by conservation of momentum arguments is:

$$T = \frac{m_0 - m_f}{\dot{m}}$$

where $m_f$ is the final "dry mass" of the rocket, 
once all the fuel is burned.

1. Calculate the burn time $T$ for the first stage of
the new Saturn V rocket that NASA is planning
to use to carry the Apollo capsule.

In [ ]:
def burn_time(m0: float, mf: float, mdot: float) -> float:
    """
    Compute rocket burn time.

    Parameters
    ----------
    m0 : float
        Initial wet mass (kg)
    mf : float
        Final dry mass (kg)
    mdot : float
        Fuel burn rate (kg/s)

    Outputs
    -------
    float
        Burn time T (seconds)
    """
    if m0 <= mf:
        raise ValueError("Wet mass must be greater than dry mass")

    if mdot <= 0:
        raise ValueError("Burn rate must be positive")

    T = (m0 - mf) / mdot
    return T


# burn time
T4 = burn_time(wetMSV, dryMSV, burnRateSV)

print(f"Burn time T = {T4:.2f} seconds")

2. Define a function that returns the change in velocity $\Delta v(t)$.
It should take all necessary arguments, meaning: the time, 
wet mass, dry mass, burn rate, exhaust speed, and gravitational constant $g$. 
Have it return the change in velocity.
Note that the rocket eventually runs out of fuel, 
so you should design your function carefully so that 
$\Delta v$ eventually becomes zero after all fuel is spent.
Again, document and guard your function according to best practices.

In [ ]:
def delta_v(t: Number, m0: float, mf: float,  mdot: float, ve: float, g: float = sc.g) -> Number:
    """
    Compute change in velocity Δv(t) for a rocket using the
    Tsiolkovsky rocket equation (what a flex)

    Parameters
    ----------
    t : float or np.ndarray
        Time since launch (seconds)
    m0 : float
        Initial wet mass (kg)
    mf : float
        Final dry mass (kg)
    mdot : float
        Burn rate (kg/s)
    ve : float
        Exhaust velocity (m/s)
    g : float
        Gravitational acceleration (m/s^2)

    Outputs
    -------
    float or np.ndarray
        Velocity change Δv(t)
    """

    if m0 <= mf:
        raise ValueError("m0 must be larger than mf")

    if mdot <= 0:
        raise ValueError("Burn rate must be positive")

    # Burn time
    T = (m0 - mf) / mdot

    # convert t to array for vector support
    t = np.asarray(t)

    # mass as function of time
    m = m0 - mdot * np.minimum(t, T)

    # Δv equation
    dv = ve * np.log(m0 / m) - g * t

    # after burnout thrust stops
    dv = np.where(t > T, ve * np.log(m0 / mf) - g * t, dv)

    return dv

#t = np.linspace(0,200,500)

#dV = delta_v(t , wetMSV, dryMSV, burnRateSV, exVelSatV)

#print(dV[:5])



3. Compute the altitude of the rocket at the end of the burn.
Do this by numerically integrating from $t=0$ (launch) to $t=T$ (burnout).
Use the `scipy.integrate.quad()` function.

In [ ]:
bT = (wetMSV - dryMSV) / burnRateSV

uhoh, err = quad(delta_v, 0, bT, args=(wetMSV, dryMSV, burnRateSV, exVelSatV))

print(f"Burn time bT = {bT:.2f} s")
print(f"Altitude at burnout uhoh = {uhoh:.2f} m")
print(f"Estimated integration error = {err:.2e}")

4. Just last week, NASA got the test results back from the first prototype
of Saturn V. They found that it burned for about 160 seconds
and lifted the system to an altitude of about 70km.
Director Kranz has asked you to compare your calculations for 
$T$ and $h$ to the testing data.
You should offer some explanations about why your calculation
is an over or under estimate, e.g. neglecting drag.

**Your Answer Goes in the markdown cell below:**

When comparing our calculated altitude to the test results we see that our altitude calculation was a little high and our burn time was a little shorter than reality.
However, when we consider that our model neglects the impact of effects like drag in the atmosphere, as well as holding some values constant, like the escape velocity 
and g these calculations makes sense. Drag would slow the rocket, thus making our calculated altitude lower than a real world test. Not having little g and our escape
velocity as constants also slightly decreases our calculated burn time. In reality these numbers would be decreasing throughout the launch.